# 06 · Clustering

> **Module 4 · Partitioning & Clustering · Lectures 4.6–4.7**

In this notebook you will:

- Create a clustered table
- See how clustering order affects the bytes read
- Add and remove clustering on an existing table
- Compare clustering against partitioning

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. What clustering does

BigQuery stores data in **blocks**. Each block has sort keys for up to 4 clustering
columns. A query that filters on a clustering column can skip entire blocks.

| | Partitioning | Clustering |
|---|---|---|
| Granularity | One per day (or hour) | Every value combination |
| Limit | 10,000 partitions | 4 columns |
| Ordering | Coarse | Fine (lexicographic) |
| Expiration | ✅ Automatic | ❌ Not possible |
| Storage overhead | None | 5–20% |
| Best for | Time filters, retention | Repeated filters on the same columns |

In [ ]:
# A clustered table — up to 4 columns, in priority order
clustered = bigquery.Table(f"{PROJECT_ID}.curated.orders_clustered", schema=[
    bigquery.SchemaField("order_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("total_amount", "NUMERIC"),
    bigquery.SchemaField("status", "STRING"),
    bigquery.SchemaField("region", "STRING"),
    bigquery.SchemaField("channel", "STRING"),
])
clustered.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="order_date",
    require_partition_filter=True,
)
clustered.clustering_fields = ["customer_id", "status", "region"]
clustered.description = "Partitioned by date, clustered by customer, status, region"
clustered = client.create_table(clustered, exists_ok=True)

print(f"✓ {clustered.full_table_id}")
print(f"  partition:  {clustered.time_partitioning.field}")
print(f"  clustering: {clustered.clustering_fields}")

In [ ]:
# Fill it with the same data as the partitioned table
job_config = bigquery.QueryJobConfig(
    destination=f"{PROJECT_ID}.curated.orders_clustered",
    write_disposition="WRITE_TRUNCATE",
)

job = client.query(
    f"SELECT * FROM `{PROJECT_ID}.curated.orders_part`",
    job_config=job_config,
)
job.result()
print(f"✓ Copied {job.output_rows:,} rows")

## 2. Measure the clustering benefit

Compare the bytes read with a clustering filter against one without. On a small
table the difference is negligible — the benefit scales with table size.

In [ ]:
T = f"`{PROJECT_ID}.curated.orders_clustered`"

VARIANTS = {
    "partition only":              f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15'",
    "+ cluster col 1 (customer)":  f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND customer_id = 'c0007'",
    "+ cluster col 2 (status)":     f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND status = 'active'",
    "+ cluster col 3 (region)":     f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND region = 'EMEA'",
    "+ col 1 and col 2":            f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND customer_id = 'c0007' AND status = 'active'",
    "non-clustered col (channel)":  f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND channel = 'web'",
}

print(f"{'variant':<34} {'bytes':>14} {'est. cost':>12}")
print("-" * 62)
for label, sql in VARIANTS.items():
    est = estimate_cost(sql)
    print(f"{label:<34} {est['bytes_processed']:>14,} ${est['estimated_cost_usd']:>11.6f}")

In [ ]:
# The clustering benefit metric, from the job metadata
job = client.query(VARIANTS["+ cluster col 1 (customer)"])
job.result()

if job.query_plan:
    for step in job.query_plan:
        # BigQuery reports the clustering benefit as a percentage
        benefit = step.get("computeRatio", step.get("status", ""))
        print(f"  {step.get('name', '?'):<30} {step.get('status', '?'):<12} {benefit}")

print("\nThe console's 'Execution details' tab shows 'Clustering: N%' for each step.")
print("100% means every block was filtered by the clustering key.")

## 3. Clustering order matters

The **first** clustering column gets the biggest benefit. Order by **filter
frequency**, not by cardinality.

In [ ]:
# Rank candidate clustering columns by expected benefit
filter_stats = [
    {"column": "customer_id", "filter_count": 850, "selectivity": 0.02, "cardinality": 5000},
    {"column": "status",      "filter_count": 620, "selectivity": 0.08, "cardinality": 3},
    {"column": "region",      "filter_count": 540, "selectivity": 0.15, "cardinality": 4},
    {"column": "channel",     "filter_count": 210, "selectivity": 0.35, "cardinality": 3},
]

ranked = sorted(
    filter_stats,
    key=lambda c: c["filter_count"] * (1 - c["selectivity"]),
    reverse=True,
)

print(f"{'column':<14} {'filters':>9} {'selectivity':>12} {'rows skipped':>13} {'rank':>6}")
print("-" * 58)
for i, c in enumerate(ranked, 1):
    skipped = c["filter_count"] * (1 - c["selectivity"])
    print(f"{c['column']:<14} {c['filter_count']:>9,} {c['selectivity']:>12.2%} {skipped:>13,.0f} {i:>6}")

print("\nNote: 'status' has only 3 distinct values but is filtered often —")
print("it still outranks 'channel'. Frequency beats cardinality.")

In [ ]:
# Rebuild the table with the recommended clustering order
table = client.get_table(f"{PROJECT_ID}.curated.orders_clustered")
table.clustering_fields = ["customer_id", "status", "region"]
table = client.update_table(table, update_mode="update_fields")
print(f"✓ clustering now: {table.clustering_fields}")

print("Re-clustering happens in the background over roughly the last 10 GB,")
print("so the benefit is not instant on a large table.")

## 4. Add and remove clustering

Unlike partitioning, clustering **can** be changed after creation.

In [ ]:
# Add clustering to a table that has none
table = client.get_table(f"{PROJECT_ID}.curated.orders_part")
print(f"Before: {table.clustering_fields}")

table.clustering_fields = ["customer_id", "region"]
table = client.update_table(table, update_mode="update_fields")
print(f"After:  {table.clustering_fields}")

# Remove clustering entirely
table = client.get_table(f"{PROJECT_ID}.curated.orders_part")
table.clustering_fields = None
table = client.update_table(table, update_mode="update_fields")
print(f"Removed: {table.clustering_fields}")

# Or with SQL
run_query(
    f"ALTER TABLE `{PROJECT_ID}.curated.orders_part` "
    "SET OPTIONS (clustering_fields = ['customer_id', 'region'])"
)
print("✓ Re-added via SQL")

## 5. Partition vs cluster — a worked comparison

Both mechanisms reduce the bytes read. They work at different granularities.

In [ ]:
COMPARISON = {
    "partition filter (date)":     f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15'",
    "cluster filter (customer)":   f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND customer_id = 'c0007'",
    "both":                        f"SELECT COUNT(*) FROM {T} WHERE order_date = DATE '2026-01-15' AND customer_id = 'c0007' AND status = 'active'",
    "neither (full table)":        f"SELECT COUNT(*) FROM {T}",
}

print(f"{'strategy':<32} {'bytes':>14} {'vs full table':>15}")
print("-" * 63)
costs = {}
for label, sql in COMPARISON.items():
    est = estimate_cost(sql)
    costs[label] = est["bytes_processed"]

full = costs.get("neither (full table)", 1)
for label, sql in COMPARISON.items():
    b = costs[label]
    ratio = f"{b / full:.1%}" if full else "n/a"
    print(f"{label:<32} {b:>14,} {ratio:>15}")

## 6. Clustered table with an MV

A materialized view can be clustered independently of its base table — cluster it
on the columns the **dashboard** filters on.

In [ ]:
mv = bigquery.Table(f"{PROJECT_ID}.mart.mv_daily_revenue")
mv.mview_query = f"""
    SELECT
      order_date,
      region,
      channel,
      COUNT(*) AS order_count,
      SUM(total_amount) AS revenue
    FROM `{PROJECT_ID}.curated.orders_clustered`
    GROUP BY order_date, region, channel
"""
mv.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY, field="order_date"
)
mv.clustering_fields = ["region", "channel"]
mv = client.create_table(mv, exists_ok=True)
print(f"✓ {mv.full_table_id}")
print(f"  clustering: {mv.clustering_fields}")

In [ ]:
df = run_to_dataframe(
    f"""
    SELECT order_date, region, channel, order_count, revenue
    FROM `{PROJECT_ID}.mart.mv_daily_revenue`
    ORDER BY order_date DESC, revenue DESC
    LIMIT 10
    """
)
display(df)

## 7. The rules

- **Maximum 4 clustering columns.**
- Clustering does **not** apply to external tables, views, or logical views.
- Storage overhead: **5–20%**.
- Re-clustering is automatic and happens in the background over roughly the last
  10 GB of the table — so the benefit is not instant.
- The benefit scales with table size: negligible under 1 GB, 50–80% above 100 GB.
- **Order by filter frequency**, not cardinality.

## 🔑 Takeaways

- Clustering sorts data by up to **4 columns**, letting BigQuery skip blocks.
- **The first clustering column gets the biggest benefit.**
- **Order by filter frequency, not cardinality.**
- Clustering **can** be changed after creation; partitioning cannot.
- Use **both**: partition for time and retention, cluster for repeated filters.
- Watch the `Clustering: N%` metric in the execution details.

---

## 📚 Next

[07 · Views and Materialized Views](./07-views-and-materialized-views.ipynb)